# Automatic Road Surface Defect Detection - Yolov8

### Mục lục quy trình dự án:
- **0. Chuẩn bị môi trường (Environment Setup)**
- **1. Import Dataset (Nạp và kiểm tra dữ liệu)**
- **2. Khám phá dữ liệu (Exploratory Data Analysis - EDA)**
- *3. Tiền xử lý & Chia tập Train / Val / Test (bước tiếp theo)*
- *4. Huấn luyện mô hình (Model Training)*
- *5. Đánh giá & Suy luận (Evaluation & Inference)*

## 0. Chuẩn bị môi trường (Environment Setup)

Trong phần này, chúng ta sẽ:
1. Cấu hình và cài đặt các thư viện cần thiết (`ultralytics`, `torch`, `opencv-python`, `matplotlib`, `seaborn`, `pandas`, `pyyaml`).
2. Kiểm tra phần cứng và thiết bị tăng tốc tính toán (NVIDIA CUDA / Apple Silicon MPS / CPU).
3. Cố định Random Seed để đảm bảo tính tái lập (Reproducibility) của các thí nghiệm.

In [ ]:
# 0.1. Cài đặt các thư viện cần thiết (bỏ comment nếu chạy lần đầu hoặc trên Google Colab / Kaggle)
# !pip install -q ultralytics opencv-python matplotlib seaborn pandas pyyaml pillow tqdm

In [ ]:
# 0.2. Import các thư viện lõi
import os
import sys
import random
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from tqdm import tqdm
import yaml

# Cấu hình style vẽ đồ thị
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10

# Kiểm tra PyTorch và cấu hình thiết bị phần cứng
import torch

print(f"Python Version  : {sys.version.split()[0]}")
print(f"PyTorch Version : {torch.__version__}")
print(f"OpenCV Version  : {cv2.__version__}")

# Tự động phát hiện thiết bị tối ưu (CUDA GPU / Apple Silicon MPS / CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = torch.cuda.get_device_name(0)
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    device_name = "Apple Silicon (MPS)"
else:
    device = torch.device("cpu")
    device_name = "CPU"

print(f"🚀 Sử dụng thiết bị tính toán: {device} ({device_name})")

In [ ]:
# 0.3. Cố định Random Seed để đảm bảo tính tái lập (Reproducibility)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"🌱 Random seed đã được thiết lập: {seed}")

set_seed(42)

## 1. Import Dataset

Trong phần này, chúng ta sẽ:
1. Định nghĩa đường dẫn tới thư mục chứa dữ liệu ảnh (`Data/dts/images/`) và nhãn chuẩn YOLO (`Data/dts/labels-YOLO/`).
2. Khai báo danh mục các lớp khuyết tật cần phát hiện:
   - **0: Pothole (Ổ gà)**
   - **1: Crack (Vết nứt mặt đường)**
   - **2: Manhole (Nắp cống)**
3. Nạp danh sách file và kiểm tra tính toàn vẹn (Integrity Check) giữa hình ảnh và file nhãn tương ứng.

In [ ]:
# 1.1. Cấu hình đường dẫn và thông tin các lớp đối tượng (Classes)
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "Data" / "dts"
IMAGES_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels-YOLO"

# Định nghĩa danh sách các lớp đối tượng (Classes mapping)
CLASSES = {
    0: "Pothole",   # Ổ gà
    1: "Crack",     # Vết nứt
    2: "Manhole"    # Nắp cống
}

# Bảng màu sắc đại diện cho từng lớp (RGB)
CLASS_COLORS = {
    0: (230, 57, 70),    # Đỏ cam (Pothole)
    1: (244, 162, 97),   # Cam vàng (Crack)
    2: (42, 157, 143)    # Xanh ngọc (Manhole)
}

print(f"📁 Thư mục ảnh: {IMAGES_DIR} (Tồn tại: {IMAGES_DIR.exists()})")
print(f"📁 Thư mục nhãn: {LABELS_DIR} (Tồn tại: {LABELS_DIR.exists()})")
print(f"🏷️  Các lớp phân loại: {CLASSES}")

In [ ]:
# 1.2. Quét danh sách file và kiểm tra tính toàn vẹn của Dataset (Integrity Check)
image_extensions = (".jpg", ".jpeg", ".png", ".bmp")
image_files = sorted([f for f in IMAGES_DIR.glob("*") if f.suffix.lower() in image_extensions])
label_files = sorted(list(LABELS_DIR.glob("*.txt")))

print(f"📊 Tổng số hình ảnh tìm thấy: {len(image_files)}")
print(f"📊 Tổng số tệp nhãn tìm thấy: {len(label_files)}")

# Kiểm tra độ khớp giữa file ảnh và file nhãn (stem matching)
image_stems = {f.stem for f in image_files}
label_stems = {f.stem for f in label_files}

matched = image_stems.intersection(label_stems)
missing_labels = image_stems - label_stems
missing_images = label_stems - image_stems

print(f"✅ Số cặp ảnh - nhãn hợp lệ: {len(matched)}")
if missing_labels:
    print(f"⚠️ Số ảnh chưa có file nhãn: {len(missing_labels)}")
if missing_images:
    print(f"⚠️ Số file nhãn không có ảnh tương ứng: {len(missing_images)}")

## 2. Khám phá dữ liệu (Exploratory Data Analysis - EDA)

Trong phần khám phá dữ liệu này, chúng ta sẽ thực hiện phân tích chuyên sâu:
1. **Thống kê số lượng nhãn**: Đếm tổng số Bounding Box theo từng loại lỗi (Pothole, Crack, Manhole), tính tỷ lệ phần trăm phân bố và số lượng ảnh background (ảnh không lỗi).
2. **Trực quan hóa phân bố nhãn**: Sử dụng biểu đồ trực quan hóa số lượng đối tượng và tần suất số lỗi xuất hiện trên mỗi ảnh.
3. **Phân tích hình học Bounding Box**: Khảo sát kích thước chiều rộng, chiều cao và tỷ lệ khung hình (Aspect Ratio) của các Bounding Box.
4. **Trực quan hóa ảnh thực tế**: Xây dựng hàm hiển thị mẫu ảnh kèm Bounding Box chuẩn YOLO để kiểm tra trực quan chất lượng gán nhãn.

### 2.1. Thống kê và phân tích phân bố các lớp đối tượng (Class Distribution)

In [ ]:
records = []
images_info = []

for img_path in tqdm(image_files, desc="Đang đọc và phân tích nhãn"):
    lbl_path = LABELS_DIR / f"{img_path.stem}.txt"
    obj_count = 0
    class_counts = {c: 0 for c in CLASSES.keys()}
    
    if lbl_path.exists() and lbl_path.stat().st_size > 0:
        with open(lbl_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls_id = int(parts[0])
                    x_c, y_c, w, h = map(float, parts[1:])
                    records.append({
                        "image": img_path.name,
                        "class_id": cls_id,
                        "class_name": CLASSES.get(cls_id, f"Unknown ({cls_id})"),
                        "x_center": x_c,
                        "y_center": y_c,
                        "width": w,
                        "height": h,
                        "area": w * h,
                        "aspect_ratio": w / h if h > 0 else 0
                    })
                    obj_count += 1
                    if cls_id in class_counts:
                        class_counts[cls_id] += 1
                        
    images_info.append({
        "image_name": img_path.name,
        "num_objects": obj_count,
        **{f"count_{CLASSES[c]}": class_counts[c] for c in CLASSES}
    })

df_objects = pd.DataFrame(records)
df_images = pd.DataFrame(images_info)

print(f"\n📈 Tổng số đối tượng (Bounding Boxes) được gán nhãn: {len(df_objects)}")
print("\n--- Bảng thống kê chi tiết theo từng lớp khuyết tật ---")
summary = df_objects["class_name"].value_counts().reset_index()
summary.columns = ["Tên lớp", "Số lượng"]
summary["Tỷ lệ (%)"] = (summary["Số lượng"] / len(df_objects) * 100).round(2)
print(summary.to_string(index=False))

empty_images = df_images[df_images["num_objects"] == 0]
print(f"\n🖼️ Số lượng ảnh không chứa khuyết tật (Background images): {len(empty_images)}")

### 2.2. Trực quan hóa phân bố đối tượng bằng biểu đồ


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Biểu đồ 1: Số lượng Bounding Box cho từng lớp
palette = ["#E63946", "#F4A261", "#2A9D8F"]
barplot = sns.barplot(data=summary, x="Tên lớp", y="Số lượng", palette=palette, ax=axes[0])
axes[0].set_title("Phân bố số lượng đối tượng theo từng lớp khuyết tật", fontsize=12, fontweight="bold", pad=10)
axes[0].set_xlabel("Lớp khuyết tật", fontsize=10)
axes[0].set_ylabel("Số lượng Bounding Box", fontsize=10)
for p in barplot.patches:
    barplot.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha="center", va="bottom", fontsize=10, xytext=(0, 3), textcoords="offset points")

# Biểu đồ 2: Phân bố số khuyết tật trên mỗi bức ảnh
max_objs = int(df_images["num_objects"].max())
sns.histplot(df_images["num_objects"], bins=range(0, max_objs + 2), color="#457B9D", discrete=True, ax=axes[1])
axes[1].set_title("Phân bố số lượng khuyết tật trên mỗi bức ảnh", fontsize=12, fontweight="bold", pad=10)
axes[1].set_xlabel("Số lượng khuyết tật / ảnh", fontsize=10)
axes[1].set_ylabel("Số lượng ảnh", fontsize=10)

plt.tight_layout()
plt.show()

### 2.3. Phân tích hình học Bounding Box (Kích thước width & height chuẩn hóa)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Biểu đồ phân tán kích thước width vs height của bounding box theo từng class
sns.scatterplot(data=df_objects, x="width", y="height", hue="class_name", alpha=0.6, palette=palette, ax=axes[0])
axes[0].set_title("Kích thước chuẩn hóa của Bounding Boxes (Width vs Height)", fontsize=12, fontweight="bold", pad=10)
axes[0].set_xlabel("Normalized Width", fontsize=10)
axes[0].set_ylabel("Normalized Height", fontsize=10)
axes[0].legend(title="Lớp")

# Biểu đồ hộp diện tích bounding box (Area = Width * Height)
sns.boxplot(data=df_objects, x="class_name", y="area", palette=palette, ax=axes[1])
axes[1].set_title("Phân bố diện tích Bounding Box theo từng lớp", fontsize=12, fontweight="bold", pad=10)
axes[1].set_xlabel("Lớp khuyết tật", fontsize=10)
axes[1].set_ylabel("Diện tích chuẩn hóa (Area)", fontsize=10)

plt.tight_layout()
plt.show()

### 2.4. Xây dựng hàm hiển thị ảnh kèm Bounding Box chuẩn YOLO


In [ ]:
def draw_yolo_boxes(image_path, label_path):
    """
    Đọc ảnh và vẽ các Bounding Box từ file nhãn YOLO tương ứng.
    """
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h_img, w_img, _ = img.shape
    
    if not label_path.exists():
        return img
        
    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cls_id = int(parts[0])
                x_c, y_c, w, h = map(float, parts[1:])
                
                # Chuyển đổi tọa độ chuẩn hóa YOLO sang tọa độ pixel (x1, y1, x2, y2)
                x1 = int((x_c - w / 2) * w_img)
                y1 = int((y_c - h / 2) * h_img)
                x2 = int((x_c + w / 2) * w_img)
                y2 = int((y_c + h / 2) * h_img)
                
                # Đảm bảo tọa độ nằm trong biên ảnh
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w_img, x2), min(h_img, y2)
                
                color = CLASS_COLORS.get(cls_id, (255, 255, 255))
                label_name = CLASSES.get(cls_id, f"Class {cls_id}")
                
                # Vẽ Bounding Box
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                
                # Vẽ nhãn chữ kèm nền màu
                (text_w, text_h), baseline = cv2.getTextSize(label_name, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
                cv2.rectangle(img, (x1, max(0, y1 - text_h - 4)), (x1 + text_w + 4, y1), color, -1)
                cv2.putText(img, label_name, (x1 + 2, y1 - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
                
    return img

print("✅ Đã định nghĩa hàm draw_yolo_boxes thành công!")

### 2.5. Hiển thị ngẫu nhiên các mẫu ảnh thực tế kèm nhãn khuyết tật

In [ ]:
# 2.5. Hiển thị ngẫu nhiên các mẫu ảnh thực tế kèm nhãn khuyết tật
def show_sample_grid(num_samples=6, cols=3):
    # Lấy các ảnh có chứa ít nhất 1 đối tượng khuyết tật
    valid_images = [
        img for img in image_files 
        if (LABELS_DIR / f"{img.stem}.txt").exists() and (LABELS_DIR / f"{img.stem}.txt").stat().st_size > 0
    ]
    samples = random.sample(valid_images, min(num_samples, len(valid_images)))
    
    rows = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))
    axes = np.array(axes).reshape(-1)
    
    for idx, img_path in enumerate(samples):
        lbl_path = LABELS_DIR / f"{img_path.stem}.txt"
        annotated_img = draw_yolo_boxes(img_path, lbl_path)
        axes[idx].imshow(annotated_img)
        axes[idx].set_title(f"{img_path.name}", fontsize=10)
        axes[idx].axis("off")
        
    # Ẩn các ô trống nếu số sample không lấp đầy hàng cuối
    for j in range(len(samples), len(axes)):
        axes[j].axis("off")
        
    plt.suptitle("Một số hình ảnh mẫu với nhãn khuyết tật (Pothole, Crack, Manhole)", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

show_sample_grid(num_samples=6, cols=3)